In [ ]:
import sys
import pathlib
import os
import json

# Add parent directory to Python path
sys.path.append(str(pathlib.Path.cwd().parent.parent))

# from src.vector_storage.milvus_client import MilvusStorage
from src.vector_storage.milvus_client import MilvusVectorClient
from src.retrieve.grok_retrieval_service import main
from groq import Groq

from typing import List, Dict, Any, Optional, Callable
from logging import Logger

In [ ]:
groq_key=os.environ["GROQ_API_KEY"]

In [ ]:
client = Groq(api_key=groq_key)

In [ ]:
# List of supporting models

models = client.models.list()

for model in models.data:
    print(model.id)

In [ ]:
coll_name = "funds_collection"
milvus_client = MilvusVectorClient(
    logger=Logger('Preparing Evalset'),
    collection_name=coll_name,
    dim=768,
    host='localhost',
    port='19530'
)

In [ ]:
info = milvus_client.client.describe_collection("funds_collection")
print(info)

In [ ]:
res = milvus_client.query_by_document_id(document_id = "91dffa0d-9183-447d-b11f-aaca1e7afb50") # change document id as per your dataset
for c in res:
    print(c["chunk_id"])
    print(c["text"])

In [ ]:
milvus_client.peek_chunks()

In [ ]:
def generate_answer(context: str, query: str) -> str:
    prompt = f"""
    You are helpful, knowledgeable assistant preparing evaluation set for evaluating llm models.
    Answer the user's question naturally, as a human expert would explain it in a conversation.
    Use the provided context as the primary source of information.

    Context:
    {context}

    Question:
    {query}

    Instructions:
    - Answer the question using the provided context.
    - Do not invent information that is not supported by the context.
    - If the context does not contain enough information to answer, say so.
    - Give a concise and accurate answer.
    """
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
    )

    answer = response.choices[0].message.content
    return answer

In [ ]:
def build_qa_case_from_document(
    question: str,
    document_id: str,
    milvus_client: MilvusVectorClient,
    llm_generate_fn: Optional[Callable[[str, str], str]] = None,
    max_context_chars: int = 20000,
) -> Dict[str, Any]:
    """
    Retrieve all chunks for a document_id and synthesize an answer to
    `question` using only those chunks as context.
 
    Args:
        question: The question to answer.
        document_id: Document whose chunks form the context.
        milvus_client: An initialized MilvusVectorClient.
        llm_generate_fn: Optional callable (question, context) -> answer_str.
            Defaults to an Anthropic API call. Pass your own to use a
            different provider/model, or to reuse a shared client instance
            instead of constructing a new one per call.
        max_context_chars: Safety cap on how much chunk text gets sent to
            the LLM, in case a document has an unusually large number of
            chunks. Chunks are kept in chunk_order and truncated from the
            end if this limit is hit.
 
    Returns:
        {
            "question": str,
            "document_id": str,
            "expected_chunks": List[Dict],   # raw chunk records, ordered by chunk_order
            "expected_context": str,         # chunks concatenated into one context blob
            "expected_answer": str,          # LLM-generated answer grounded in context
        }
        If no chunks are found, "answer" will explain that and "chunks"/
        "context" will be empty - callers should check for this rather
        than assuming a populated result.
    """
    result: Dict[str, Any] = {
        "question": question,
        "document_id": document_id,
        "expected_chunks": [],
        "expected_context": "",
        "expected_answer": "",
    }
 
    # 1. Retrieve chunks
    chunks = milvus_client.query_by_document_id(document_id)
    if not chunks:
        print(f"No chunks found for document_id={document_id}")
        result["expected_answer"] = (
            "No content found for this document_id - cannot generate an answer."
        )
        return result
 
    result["expected_chunks"] = chunks
 
    # 2. Build context, respecting the char cap (chunks already ordered
    #    by chunk_order inside query_by_document_id)
    context_parts: List[str] = []
    running_len = 0
    for c in chunks:
        text = c.get("text", "")
        if not text:
            continue
        if running_len + len(text) > max_context_chars:
            print(
                f"Context truncated at {running_len} chars for document_id="
                f"{document_id} (had {len(chunks)} chunks)"
            )
            break
        context_parts.append(text)
        running_len += len(text)
 
    context = "\n\n".join(context_parts)
    result["expected_context"] = context
 
    if not context:
        result["expected_answer"] = "Chunks were found but contained no text - cannot generate an answer."
        return result
 
    # 3. Generate the answer
    try:
        result["expected_answer"] = generate_answer(context=context, query=question)
    except Exception as e:
        print(f"LLM answer generation failed for document_id={document_id}: {e}")
        result["expected_answer"] = ""
 
    return result
 

In [ ]:
question="What telangana government has done on awareness for innovation?" # Frame question as per your documents

In [ ]:
case = build_qa_case_from_document(
        question=question,
        document_id="91dffa0d-9183-447d-b11f-aaca1e7afb50", # Change the document id as per your dataset
        milvus_client=milvus_client,
    )

In [ ]:
case

In [ ]:
output_response = main(query=question)

In [ ]:
output_response.keys()

In [ ]:
case["output_chunks"] = output_response.get("output_chunks")
case["output_answer"] = output_response.get("answer")

In [ ]:
case = [case]
len(case)

In [ ]:
# change path as per your project storage, if needed
with open("../../eval_dataset/evaluation_set.json", "w", encoding="utf-8") as f:
    json.dump(case, f, ensure_ascii=False, indent=4,  default=str)